In [11]:
import numpy as np

#a
states = ["D", "M", "E"]
grades = ["VG", "G", "S", "U"]

pi = np.array([1/3, 1/3, 1/3])
A = np.array([[0.0,0.5,0.5],
             [0.5,0.25,0.25],
             [0.5,0.25,0.25]])
B = np.array([[0.10, 0.20, 0.40, 0.30],
              [0.15, 0.25, 0.50, 0.10],
              [0.20, 0.30, 0.40, 0.10]])

for row in A:
    print("row sum A: ", row.sum())
for row in B:
    print("row sum B: ", row.sum())


obs_names = ["VG","VG","S","G","G","S","G","G","U","G","G","S"]
o = [grades.index(g) for g in obs_names]
T = len(o)
N = len(states)

#b
delta = np.zeros((T,N))
psi = np.zeros((T,N), dtype=int)
delta[0][0] = pi[0] * B[0][0]
delta[0][1] = pi[1] * B[1][0]
delta[0][2] = pi[2] * B[2][0]

#c
for t in range (1,T):
    for j in range(N):
        best_score = 0
        best_prev = 0
        for i in range(N):
            score = delta[t-1][i] * A[i][j]
            if score > best_score:
                best_score = score
                best_prev = i
        delta[t][j] = B[j][o[t]] * best_score
        psi[t][j] = best_prev

'''the role of psi is to store where the best score came from, so we can rebuild the path. without it we would only know the score but not where it came from'''

print("delta1", delta[0]); print("delta2", delta[1])
print("psi2", psi[1])

'''by hand:
delta1(D) = 1/3 * 0.1 = 0.0(3)
delta1(M) = 1/3 * 0.15= 0.05
delta1(E) = 1/3 * 0.2 = 0.0666667

delta2(D)
from D: 0.0(3) * 0 = 0
from M: 0.05 * 0.5 = 0.025
from E: 0.0667 x 0.5 = 0.0333 <- biggest

delta2(D) = 0.1 x 0.0333 = 0.003333
psi2(D) = E

delta2(M)
from D: 0.03333 x 0.5 = 0.01667 <- biggest
from M: 0.05 x 0.25 = 0.0125
from E: 0.0667 x 0.25 = 0.01667 <- biggest

delta2(M) = 0.15 x 0.01667 = 0.0025
psi2(M) = D

delta2(E)
from D: 0.0333 x 0.5 = 0.01667 <- biggest
from M: 0.05 x 0.25 = 0.0125
from E: 0.0667 x 0,25 = 0.01667 <- biggest

delta2(E) = 0.2 x 0.01667 = 0.00333
psi2(E) = D

delta 1 = (0.(3) , 0.05, 0.0666667)
delta 2 = (0.00(3), 0.0025, 0.00(3))

best one (at test 2) is E, D tied with D, E
''';

#e
last = np.argmax(delta[-1])
best = delta[-1, last]
path = [last]
for t in range(T-1, 0, -1):
    path.append(psi[t, path[-1]])

path = path[::-1]
print([states[i] for i in path])
print("max joint P =", best)


'''
at each step on the viterbi algorithm it keeps the best path ending in each state and extends it, so the final score belongs to one entire path.

picking the most likely difficulty separately for each test (using P(X_t | o) can give a worse or even impossible path (for example it could choose D->D which is impossible)

The value from e) is a joint probability P(x*, o) (meaning this path and these grades together), it is not the chance that the path is right given the grades.
''';

row sum A:  1.0
row sum A:  1.0
row sum A:  1.0
row sum B:  1.0
row sum B:  1.0
row sum B:  1.0
delta1 [0.03333333 0.05       0.06666667]
delta2 [0.00333333 0.0025     0.00333333]
psi2 [2 0 0]
['D', 'E', 'D', 'E', 'D', 'M', 'D', 'E', 'D', 'E', 'D', 'M']
max joint P = 2.1093750000000005e-11
